In [1]:
%load_ext autoreload
%autoreload 2

In [11]:
import os
# TODO: port the benchmark suite to the new class structure, then remove this file and the corresponding test file
from mex.task import RepetitionDecreaseTask, SegmentBoundaryTask, IDyOMConsistencyTask
from mex.metadata import SessionIDyOMMetaData, SalamiMetaData
from mex.dataset_provider import DatasetProvider
from mex.baseline.audioic import AudioIC

# Task metadata

In [19]:
dataset_provider = DatasetProvider(
    cache_dir=None, # NOTE: will cache datasets in platform specific location, e.g. ~/.cache/mex on linux
    hf_root="muthissar/music_surprisal_benchmarks",
    hf_token=os.environ.get("HF_ACCESS_TOKEN"),
)
salami_metadata = SalamiMetaData(
    dataset_provider=dataset_provider,
    salami_root="/dir/to/salami"
)

session_idyom_metadata = SessionIDyOMMetaData(
    dataset_provider=dataset_provider,
    idyom_dat_dir="/dir/to/idyom.dat",
    midi_dir="/dir/to/midi/files"
)

## Preprocess
(skip if using already cached datasets, or loading from hub)

preprocess datasets and stores cached versions in cache_dir

In [ ]:
salami_metadata.preprocess()
session_idyom_metadata.preprocess()

## Load precomputed metadata datasets

In [20]:
# get datasets and parse
salami_metadata_ds = salami_metadata.load()
session_metdata_ds = session_idyom_metadata.load()
# or from non_standard location, e.g. local or hfhub
sessisession_metdata_ds_2 = dataset_provider.get_ds(dataset_provider.cache_dir + "/session_idyom")
assert sessisession_metdata_ds_2.features == session_metdata_ds.features

In [42]:
# Running tasks

## IDyOM consistency

In [21]:
# Get session paired audio, idyom dataset from the hub
session_ds = dataset_provider.get_ds(dataset_provider.cache_dir + "/session_idyom")
session_ds.features

{'id': Value('int64'),
 'onset': List(Value('float64')),
 'audios': {'Acoustic Grand Piano': Value('binary'),
  'Acoustic Guitar (nylon)': Value('binary'),
  'Pad 1 (new age)': Value('binary'),
  'Synth Voice': Value('binary'),
  'Trumpet': Value('binary')},
 'idyom_ic': List(Value('float64')),
 'idyom_entropy': List(Value('float64'))}

In [24]:
# optionally filter dataset (for faster testing)
n_audios = 20
session_ds_small = session_ds.select(range(n_audios))

In [25]:
# load baseline model
baseline = AudioIC(device='cpu')

In [26]:
ic_ds = baseline.ic(session_ds_small)

Computing IC: 100%|██████████| 20/20 [05:21<00:00, 16.09s/it]


In [27]:
ic_ds.features

{'id': Value('int64'),
 'instrument': Value('string'),
 'ic': List(Value('float32'))}

In [58]:
# load only the relevant columns for idyom IC correlation from session_ds (idyom_ic)
idyom_ds = session_ds_small.select_columns(["id", "onset",  "idyom_ic"]) 
idyom_consistency_task = IDyOMConsistencyTask(
    periode_frames=AudioIC.periode_frames(),
)
res = idyom_consistency_task.run_task(
    midm=ic_ds,
    metadata=session_metdata_ds,
)
print(res)

             instrument label    r_val  p_val    idyom_midm
   Acoustic Grand Piano    ic 0.504193    0.0      idyom_ic
Acoustic Guitar (nylon)    ic 0.498678    0.0      idyom_ic
        Pad 1 (new age)    ic 0.534058    0.0      idyom_ic
            Synth Voice    ic 0.479777    0.0      idyom_ic
                Trumpet    ic 0.543278    0.0      idyom_ic
               combined    ic 0.470411    0.0      idyom_ic
   Acoustic Grand Piano    ic 0.439009    0.0 idyom_entropy
Acoustic Guitar (nylon)    ic 0.427731    0.0 idyom_entropy
        Pad 1 (new age)    ic 0.431896    0.0 idyom_entropy
            Synth Voice    ic 0.373372    0.0 idyom_entropy
                Trumpet    ic 0.457446    0.0 idyom_entropy
               combined    ic 0.389537    0.0 idyom_entropy


In [59]:
# running the experiment on all values in the dataset
# TODO: store in hub
ic_ds = dataset_provider.get_ds("cache/metrics/session_audioic_ic")
idyom_ds = session_ds.select_columns(["id", "onset",  "idyom_ic"]) 
# idyom_ds = idyom_ds.rename_column("idyom_ic", "idyom_val")
result = idyom_consistency_task.run_task(
    midm=ic_ds,
    metadata=session_metdata_ds,
)
print(result)

             instrument label    r_val  p_val    idyom_midm
   Acoustic Grand Piano    ic 0.504193    0.0      idyom_ic
Acoustic Guitar (nylon)    ic 0.498678    0.0      idyom_ic
        Pad 1 (new age)    ic 0.534058    0.0      idyom_ic
            Synth Voice    ic 0.479777    0.0      idyom_ic
                Trumpet    ic 0.543278    0.0      idyom_ic
               combined    ic 0.470411    0.0      idyom_ic
   Acoustic Grand Piano    ic 0.439009    0.0 idyom_entropy
Acoustic Guitar (nylon)    ic 0.427731    0.0 idyom_entropy
        Pad 1 (new age)    ic 0.431896    0.0 idyom_entropy
            Synth Voice    ic 0.373372    0.0 idyom_entropy
                Trumpet    ic 0.457446    0.0 idyom_entropy
               combined    ic 0.389537    0.0 idyom_entropy


## Repetition decrease

In [57]:

# Get precomputed IC values for salami dataset
salami_ic = dataset_provider.get_ds("cache/metrics/salami_audioic_ic/").remove_columns("Time")
# Get salami segment annotation metadata from local disk
repetion_decrease_task = RepetitionDecreaseTask(
    periode_frames = AudioIC.periode_frames(),
    time_offset_frame =  1 # for IC 1 for entropy 0
)
results = repetion_decrease_task.run_task(
    midm=salami_ic,
    metadata=salami_metadata_ds,
)
print(results)

Map: 100%|██████████| 1358/1358 [00:05<00:00, 255.01 examples/s]


midm_column  mode  seg_name      mean      p_value
         IC   all functions -0.047633 9.290729e-39
         IC   all lowercase -0.029830 2.409445e-72
         IC   all uppercase -0.054142 1.600254e-49
         IC first functions -0.082067 5.025470e-45
         IC first lowercase -0.075825 6.817193e-88
         IC first uppercase -0.053426 2.408463e-19


## Segment Boundary

In [56]:
segment_boundary_task = SegmentBoundaryTask(
    periode_frames = AudioIC.periode_frames(),
    time_offset_frame =  1 # for IC 1 for entropy 0
)
results = segment_boundary_task.run_task(
    midm=salami_ic,
    metadata=salami_metadata_ds,
)
print(results)

Map: 100%|██████████| 1358/1358 [00:05<00:00, 266.04 examples/s]


hierarchy_level      repetition_mode midm peak_picking_kwargs  precision   recall  f_measure
      functions                  all   IC        {"std": 5.0}   0.088948 0.545451   0.152954
      functions               change   IC        {"std": 5.0}   0.071994 0.529113   0.126743
      functions only_first_occurence   IC        {"std": 5.0}   0.039375 0.509501   0.073101
      lowercase                  all   IC        {"std": 5.0}   0.295140 0.567054   0.388220
      lowercase               change   IC        {"std": 5.0}   0.207167 0.571179   0.304054
      lowercase only_first_occurence   IC        {"std": 5.0}   0.075442 0.576996   0.133437
      uppercase                  all   IC        {"std": 5.0}   0.085152 0.551562   0.147529
      uppercase               change   IC        {"std": 5.0}   0.067990 0.537783   0.120717
      uppercase only_first_occurence   IC        {"std": 5.0}   0.036908 0.525581   0.068973
